# Hiring model on Amazon SageMaker, monitored by watsonx.governance (OpenScale) on AWS

This notebook:
1. Trains a binary classifier (scikit-learn logistic regression, trained locally) on `hiring_training_data.csv` and deploys it to a SageMaker real-time endpoint.
2. Registers the model as an **external model** in the inventory, so that it can be tracked in an AI use case.
3. Connects to **watsonx.governance as a Service on AWS** (OpenScale) and registers **Amazon SageMaker** as a machine learning provider.
4. Subscribes the endpoint, logs payload (`hiring_payload_data.csv`) and feedback (`hiring_eval_data_1.csv`).
5. Configures **Quality**, **Fairness** (`IsFemale`) and **Explainability** monitors and runs them.
6. Logs a second payload batch (`hiring_payload_data_2.csv`) for a drift story (Drift v2 is configured in the UI).

**Prerequisites**
- watsonx.governance on AWS in **Mumbai (ap-south-1)** or **N. Virginia (us-east-1)**. Other regions have no model monitoring.
- An OpenScale data mart already set up in the UI (`https://<region>.aws.data.ibm.com/aiopenscale/insights`), or Aurora PostgreSQL / Db2 credentials to create one below.
- An IBM API key: watsonx.governance UI → *Administration → Access (IAM) → Personal Access Keys*.
- AWS credentials for this notebook (uploading the model and deploying the endpoint), plus an **access key** for OpenScale with SageMaker list/describe + `sagemaker:InvokeEndpoint` (explainability scores the endpoint).
- A SageMaker execution role ARN.

## 0. Install dependencies

In [1]:
%pip install ibm-aigov-facts-client python-dotenv boto3 pandas scikit-learn "ibm-watson-openscale>=3.1.8" ibm-metrics-plugin KDEpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Configuration

In [2]:
import json, os
from dotenv import load_dotenv

# Every setting comes from the .env file next to this notebook (see .env.example)
load_dotenv(".env", override=True)

def setting(name):
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"{name} is not set; add it to .env")
    return value

# ---- AWS (SageMaker) --------------------------------------------------------
AWS_REGION = setting("AWS_REGION")                     # region where SageMaker runs
SAGEMAKER_ROLE_ARN = setting("SAGEMAKER_ROLE_ARN")
ENDPOINT_NAME = setting("ENDPOINT_NAME")

# Access key that OpenScale uses to discover and score the endpoint
OPENSCALE_AWS_ACCESS_KEY_ID = setting("OPENSCALE_AWS_ACCESS_KEY_ID")
OPENSCALE_AWS_SECRET_ACCESS_KEY = setting("OPENSCALE_AWS_SECRET_ACCESS_KEY")

# ---- watsonx.governance on AWS ----------------------------------------------
IBM_API_KEY = setting("IBM_API_KEY")
PLATFORM_URL = setting("PLATFORM_URL")                     # us-east-1: https://us-east-1.aws.data.ibm.com
OPENSCALE_URL = setting("OPENSCALE_URL")                   # us-east-1: https://us-east-1.mm.aws.governance.ibm.com
OPENSCALE_INSTANCE_ID = setting("OPENSCALE_INSTANCE_ID")   # from the Insights UI (URL / Actions > Endpoints)
SERVICE_PROVIDER_NAME = setting("SERVICE_PROVIDER_NAME")

# Only needed if no data mart exists yet (Aurora PostgreSQL or Db2): one line of JSON in .env,
# e.g. DB_CREDENTIALS='{"type": "postgresql", "hostname": "...", "port": 5432, ...}'
_db = os.getenv("DB_CREDENTIALS", "")
DB_CREDENTIALS = json.loads(_db) if _db.strip() not in ("", "None", "null") else None

# ---- Data -------------------------------------------------------------------
DATA_DIR = setting("DATA_DIR")   # folder containing the hiring CSVs
LABEL = setting("LABEL")
PROTECTED_ATTRIBUTE = setting("PROTECTED_ATTRIBUTE")

## 2. Load the hiring data

In [3]:
import pandas as pd

train_df = pd.read_csv(f"{DATA_DIR}/hiring_training_data.csv", dtype=float)
feedback_df = pd.read_csv(f"{DATA_DIR}/hiring_eval_data_1.csv", dtype=float)
payload_df_1 = pd.read_csv(f"{DATA_DIR}/hiring_payload_data.csv", dtype=float)
payload_df_2 = pd.read_csv(f"{DATA_DIR}/hiring_payload_data_2.csv", dtype=float)

FEATURES = [c for c in train_df.columns if c != LABEL]
CATEGORICAL = []   # every column in this dataset is already numerically encoded

print(train_df.shape, feedback_df.shape, payload_df_1.shape, payload_df_2.shape)
print("Label balance:", train_df[LABEL].value_counts().to_dict())
train_df.head()

(2341, 19) (110, 19) (150, 18) (150, 18)
Label balance: {0.0: 1677, 1.0: 664}


,HIRED,Age,BusinessTravel,Education,RelevantEducationLevel,JobLevel,MaritalStatus,NumCompaniesWorked,OverTime,InterviewScore,ResumeScore,RequestedBenefits,TotalWorkingYears,PreferredSkills,YearsAtCurrentCompany,RelevantExperience,JobType,SalaryExpectation,IsFemale
0,1.0,33.0,2.0,3.0,3.0,0.0,0.0,1.0,1.0,2.0,2.0,0.0,8.0,2.0,8.0,7.0,1.0,0.0,1.0
1,1.0,32.0,2.0,2.0,3.0,0.0,1.0,0.0,0.0,3.0,2.0,0.0,8.0,1.0,7.0,7.0,0.0,1.0,0.0
2,1.0,35.0,1.0,1.0,4.0,0.0,0.0,0.0,0.0,1.0,2.0,1.0,6.0,1.0,5.0,4.0,1.0,1.0,0.0
3,1.0,29.0,1.0,3.0,4.0,2.0,2.0,1.0,0.0,0.0,2.0,1.0,10.0,0.0,10.0,9.0,1.0,0.0,1.0
4,1.0,34.0,1.0,3.0,3.0,2.0,1.0,0.0,0.0,1.0,2.0,0.0,13.0,2.0,12.0,6.0,1.0,2.0,1.0


## 3. Train locally and deploy on SageMaker

The model is a scikit-learn logistic regression trained in this notebook, so no SageMaker
training job (or training-instance quota) is needed. It is deployed to a real-time endpoint on the
AWS scikit-learn serving container with the `inference.py` script below.

The endpoint accepts CSV rows (no header) and returns the same shape as the built-in Linear Learner,
`{"predictions": [{"score": p, "predicted_label": 0.0|1.0}]}`, which maps directly onto
OpenScale's `prediction_field="predicted_label"` and `probability_fields=["score"]`.

In [4]:
%%writefile inference.py
"""SageMaker inference script for the hiring model.

Scores a logistic regression with numpy from exported parameters (model.json) and
returns the same JSON shape as the built-in Linear Learner:
{"predictions": [{"score": p, "predicted_label": 0.0|1.0}]}
"""
import io
import json
import os

import numpy as np


def model_fn(model_dir):
    with open(os.path.join(model_dir, "model.json")) as f:
        return {k: np.asarray(v, dtype=float) for k, v in json.load(f).items()}


def input_fn(body, content_type):
    if isinstance(body, (bytes, bytearray)):
        body = body.decode("utf-8")
    if content_type and content_type.startswith("application/json"):
        payload = json.loads(body)
        instances = payload["instances"] if isinstance(payload, dict) else payload
        rows = [i["features"] if isinstance(i, dict) else i for i in instances]
        return np.asarray(rows, dtype=float)
    return np.loadtxt(io.StringIO(body), delimiter=",", ndmin=2)


def predict_fn(data, model):
    z = ((data - model["mean"]) / model["scale"]) @ model["coef"] + model["intercept"]
    return 1.0 / (1.0 + np.exp(-z))


def output_fn(scores, accept):
    predictions = [{"score": float(s), "predicted_label": float(s >= 0.5)} for s in scores]
    return json.dumps({"predictions": predictions}), "application/json"

Overwriting inference.py


In [5]:
import json, os, tarfile, time
import boto3
from botocore.exceptions import ClientError
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

boto_sess = boto3.Session(region_name=AWS_REGION)
s3 = boto_sess.client("s3")
sm = boto_sess.client("sagemaker")

# Train locally: no SageMaker training job (and no training-instance quota) is needed.
scaler = StandardScaler().fit(train_df[FEATURES])
clf = LogisticRegression(max_iter=1000).fit(scaler.transform(train_df[FEATURES]), train_df[LABEL])
print("Training accuracy:", round(clf.score(scaler.transform(train_df[FEATURES]), train_df[LABEL]), 3))

# Export plain parameters instead of a pickle, so the endpoint does not depend on the
# serving container having the same scikit-learn version as this notebook.
os.makedirs("sagemaker_model", exist_ok=True)
with open("sagemaker_model/model.json", "w") as f:
    json.dump({"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
               "coef": clf.coef_[0].tolist(), "intercept": clf.intercept_.tolist()}, f)
with tarfile.open("sagemaker_model/model.tar.gz", "w:gz") as tar:
    tar.add("sagemaker_model/model.json", arcname="model.json")
with tarfile.open("sagemaker_model/sourcedir.tar.gz", "w:gz") as tar:
    tar.add("inference.py")

# Upload both archives to the account's default SageMaker bucket
account_id = boto_sess.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{AWS_REGION}-{account_id}"
try:
    s3.head_bucket(Bucket=bucket)
except ClientError:
    location = {} if AWS_REGION == "us-east-1" else {"CreateBucketConfiguration": {"LocationConstraint": AWS_REGION}}
    s3.create_bucket(Bucket=bucket, **location)

s3.upload_file("sagemaker_model/model.tar.gz", bucket, "hiring-demo/model/model.tar.gz")
s3.upload_file("sagemaker_model/sourcedir.tar.gz", bucket, "hiring-demo/model/sourcedir.tar.gz")
s3_model = f"s3://{bucket}/hiring-demo/model/model.tar.gz"
s3_code = f"s3://{bucket}/hiring-demo/model/sourcedir.tar.gz"
print("Model artifact:", s3_model)

Training accuracy: 0.831
Model artifact: s3://sagemaker-us-east-1-844183293104/hiring-demo/model/model.tar.gz


In [6]:
# Takes ~5-8 minutes
# AWS-managed scikit-learn serving container; the registry account differs per region.
SKLEARN_IMAGE_ACCOUNTS = {"us-east-1": "683313688378", "ap-south-1": "720646828776"}
image_uri = (f"{SKLEARN_IMAGE_ACCOUNTS[AWS_REGION]}.dkr.ecr.{AWS_REGION}.amazonaws.com/"
             "sagemaker-scikit-learn:1.2-1-cpu-py3")

# Model and endpoint-config names must be unique, so re-running this cell creates new
# ones and points the existing endpoint at them.
model_name = f"{ENDPOINT_NAME}-{time.strftime('%Y%m%d-%H%M%S')}"
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN,
    PrimaryContainer={
        "Image": image_uri,
        "ModelDataUrl": s3_model,
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": s3_code,
            "SAGEMAKER_REGION": AWS_REGION,
        },
    },
)
sm.create_endpoint_config(
    EndpointConfigName=model_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": "ml.t2.medium",   # smallest hosting type; the endpoint bills until deleted
    }],
)

existing = sm.list_endpoints(NameContains=ENDPOINT_NAME)["Endpoints"]
if any(e["EndpointName"] == ENDPOINT_NAME for e in existing):
    sm.update_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=model_name)
else:
    sm.create_endpoint(EndpointName=ENDPOINT_NAME, EndpointConfigName=model_name)

sm.get_waiter("endpoint_in_service").wait(EndpointName=ENDPOINT_NAME)
print("Endpoint:", ENDPOINT_NAME, sm.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"])

Endpoint: hiring-model-endpoint-v3 InService


In [7]:
import json, time

sm_runtime = boto_sess.client("sagemaker-runtime")

def score(df):
    """Score a DataFrame of FEATURES; returns (predictions, response_time_ms)."""
    start = time.time()
    resp = sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="text/csv",
        Accept="application/json",
        Body=df[FEATURES].to_csv(index=False, header=False),
    )
    elapsed = int((time.time() - start) * 1000)
    return json.loads(resp["Body"].read())["predictions"], elapsed

preds, ms = score(payload_df_1.head(3))
print(ms, "ms", json.dumps(preds, indent=2))

1723 ms [
  {
    "score": 0.46325204163524386,
    "predicted_label": 0.0
  },
  {
    "score": 0.586206523005631,
    "predicted_label": 1.0
  },
  {
    "score": 0.7060609424474288,
    "predicted_label": 1.0
  }
]


## 3b. Register the model as an external model (before subscribing)

Subscribing the endpoint in OpenScale does not put it on the **External models** page, so this cell
registers the SageMaker model in the inventory with the AI Factsheets client. It runs **before** the
OpenScale sections on purpose: watsonx.governance attaches evaluation results to the model that is already
registered in an inventory when the first evaluation runs. It only needs sections 1 and 3.

The deployment identifier must be the SageMaker **endpoint name**: that is what links the OpenScale
evaluation results to the external model.

**Before you continue to section 4**, open *AI governance → External models*, find the model and use
**Track in AI use case**. Facts are only kept for a model that is tracked.

In [8]:
# %pip installs into the environment of the running kernel (!pip can target a different Python)
%pip install -q ibm-aigov-facts-client

import boto3
import requests
from ibm_aigov_facts_client import AIGovFactsClient
from ibm_aigov_facts_client.supporting_classes.factsheet_utils import DeploymentDetails, ModelDetails
from ibm_aigov_facts_client.utils.enums import Provider

load_dotenv(".env", override=True)             # pick up values added to .env since section 1 ran
IBM_ACCOUNT_ID = setting("IBM_ACCOUNT_ID")     # IBM account ID, not the AWS account ID
INVENTORY_ID = setting("INVENTORY_ID")         # there is no platform default catalog on AWS

# The facts client names the watsonx.governance region differently from the platform URL
FACTS_REGIONS = {"us-east-1": "aws_nv", "ap-south-1": "aws_mumbai"}
facts_region = next(r for key, r in FACTS_REGIONS.items() if key in PLATFORM_URL)

# On AWS the client always sets up a local MLflow experiment (in ./mlruns), even though nothing
# is traced here. It needs a name, and set_as_current_experiment lets the cell be re-run.
facts_client = AIGovFactsClient(
    api_key=IBM_API_KEY,
    account_id=IBM_ACCOUNT_ID,
    region=facts_region,
    experiment_name="hiring-external-model",
    set_as_current_experiment=True,
    enable_autolog=False,
    external_model=True)

def use_instance(instance_id):
    """Point the client, and the helpers that copied the ID when it was created, at one instance."""
    facts_client.set_service_instance_id(instance_id)
    facts_client.assets.instance_id = instance_id
    facts_client.external_model_facts.instance_id = instance_id

def visible_inventories():
    return {i.get_id(): i.to_dict().get("inventory_name") for i in facts_client.assets.list_inventories()}

# On AWS the client does not work out which watsonx.governance instance to use, and without one
# it sees no inventories. Find the instance that holds INVENTORY_ID.
inventories = visible_inventories()
if INVENTORY_ID not in inventories:
    api_url = PLATFORM_URL.replace("https://", "https://api.")
    resp = requests.get(f"{api_url}/v1/aigov/factsheet/instances",
                        headers=facts_client.assets._get_headers(), timeout=30)
    resp.raise_for_status()
    instance_ids = [r["guid"] for r in resp.json().get("resources", [])]
    print("Governance instances:", instance_ids)
    for instance_id in instance_ids:
        use_instance(instance_id)
        inventories = visible_inventories()
        if INVENTORY_ID in inventories:
            print("Using governance instance:", instance_id)
            break
    else:
        raise RuntimeError(f"INVENTORY_ID {INVENTORY_ID!r} was not found in any governance instance. "
                           f"Last inventories seen: {inventories}")
print("Saving to inventory:", inventories[INVENTORY_ID])

# The model currently behind the endpoint (its name changes every time section 3 is re-run)
sm_client = boto3.Session(region_name=AWS_REGION).client("sagemaker")
endpoint_config = sm_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointConfigName"]
sagemaker_model_name = sm_client.describe_endpoint_config(
    EndpointConfigName=endpoint_config)["ProductionVariants"][0]["ModelName"]

# Saving again fails with "Access Forbidden" once the model is tracked in an AI use case, so the
# asset ID from the first save is kept in a local file and the existing model is fetched instead.
# A redeployed or renamed endpoint has a new SageMaker model name, which is registered afresh.
EXTERNAL_MODEL_FILE = "sagemaker_model/external_model.json"

saved = {}
if os.path.exists(EXTERNAL_MODEL_FILE):
    with open(EXTERNAL_MODEL_FILE) as f:
        saved = json.load(f)

if saved.get("model_identifier") == sagemaker_model_name:
    external_model = facts_client.assets.get_model(
        model_id=saved["asset_id"], container_type="catalog", container_id=saved["container_id"])
    print("Using the external model registered earlier:", saved["asset_id"])
else:
    try:
        external_model = facts_client.external_model_facts.save_external_model_asset(
            model_identifier=sagemaker_model_name,
            name="Hiring model (SageMaker)",
            description="Logistic regression that scores job candidates, deployed on Amazon SageMaker",
            model_details=ModelDetails(
                model_type="scikit-learn",
                input_type="structured",
                algorithm="Logistic regression",
                label_type="binary",
                label_column=LABEL,
                prediction_type="binary",
                provider=Provider.AMAZON_SAGEMAKER),
            deployment_details=DeploymentDetails(
                identifier=ENDPOINT_NAME,
                name=ENDPOINT_NAME,
                deployment_type="online",
                scoring_endpoint=ENDPOINT_NAME),
            catalog_id=INVENTORY_ID)
    except KeyError:
        # The client logs a rejected save and carries on, then fails looking up a default catalog
        raise RuntimeError("The service rejected the external model. The reason is in the "
                           "log line above.") from None
    with open(EXTERNAL_MODEL_FILE, "w") as f:
        json.dump({"model_identifier": sagemaker_model_name, "asset_id": external_model.get_id(),
                   "container_id": external_model.get_container_id()}, f)

external_model.get_info()


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
2026/10/04 21:05:41 INFO : Successfully set hiring-external-model as current experiment
-----------------------------------------------------------------------------------------------------------------------------
                                                 Inventory Retrieval Started                                                 
-----------------------------------------------------------------------------------------------------------------------------
2026/10/04 21:05:41 INFO : Initiating the inventory retrieval process. Please wait...
2026/10/04 21:05:43 ERROR : No Inventories were found.

Total number of inventories: 0
Governance instances: ['20260721-1301-2811-9080-3ea091e49514']
-----------------------------------------------------------------------------------------------------------------------------
                                                 Inventory Retrieval Started                              

{'asset_id': '01a10705-5b5a-778d-b63e-910137ba2df5',
 'container_type': 'catalog',
 'container_id': '01a06d5a-9a2d-711d-8c72-d61c8769e471',
 'facts_type': 'modelfacts_user'}

## 4. Connect to OpenScale on watsonx.governance (AWS)

AWS instances authenticate with an MCSP token (`/api/rest/mcsp/apikeys/token`). Tokens expire,
so `refresh_token()` re-issues one; call it again if you get a 401 later in the demo.

In [9]:
import requests
from ibm_cloud_sdk_core.authenticators import BearerTokenAuthenticator
from ibm_watson_openscale import APIClient
from ibm_watson_openscale.supporting_classes.enums import *
from ibm_watson_openscale.supporting_classes import *
from ibm_watson_openscale.base_classes.watson_open_scale_v2 import *

def get_token():
    r = requests.post(f"{PLATFORM_URL}/api/rest/mcsp/apikeys/token",
                      json={"apikey": IBM_API_KEY}, timeout=30)
    r.raise_for_status()
    body = r.json()
    return body.get("token") or body.get("access_token")

authenticator = BearerTokenAuthenticator(bearer_token=get_token())

def refresh_token():
    authenticator.set_bearer_token(get_token())

wos_client = APIClient(authenticator=authenticator,
                       service_url=OPENSCALE_URL,
                       service_instance_id=OPENSCALE_INSTANCE_ID)
print("OpenScale SDK", wos_client.version, "->", wos_client.service_url)

OpenScale SDK 3.1.8 -> https://us-east-1.mm.aws.governance.ibm.com/openscale/20260721-1301-2811-9080-3ea091e49514


## 5. Data mart

In [10]:
data_marts = wos_client.data_marts.list().result.data_marts
if data_marts:
    data_mart_id = data_marts[0].metadata.id
    print("Using existing data mart:", data_mart_id)
elif DB_CREDENTIALS:
    db_type = DatabaseType.POSTGRESQL if DB_CREDENTIALS["type"] == "postgresql" else DatabaseType.DB2
    data_mart_id = wos_client.data_marts.add(
        name="WOS Data Mart",
        description="Hiring demo data mart",
        database_configuration=DatabaseConfigurationRequest(
            database_type=db_type,
            credentials=PrimaryStorageCredentialsLong(
                hostname=DB_CREDENTIALS["hostname"], port=DB_CREDENTIALS["port"],
                username=DB_CREDENTIALS["username"], password=DB_CREDENTIALS["password"],
                db=DB_CREDENTIALS["database"], ssl=True,
                sslmode=DB_CREDENTIALS.get("sslmode"),
                certificate_base64=DB_CREDENTIALS.get("certificate_base64")),
            location=LocationSchemaName(schema_name=DB_CREDENTIALS["schema"])),
        background_mode=False).result.metadata.id
    print("Created data mart:", data_mart_id)
else:
    raise RuntimeError("No data mart found. Set one up in the OpenScale UI or fill in DB_CREDENTIALS.")

Using existing data mart: 20260721-1301-2811-9080-3ea091e49514


## 6. Add Amazon SageMaker as a machine learning provider

In [11]:
for sp in wos_client.service_providers.list().result.service_providers:
    if sp.entity.name == SERVICE_PROVIDER_NAME:
        # Includes subscriptions for an endpoint name that is no longer used
        for sub in wos_client.subscriptions.list().result.subscriptions:
            if sub.entity.service_provider_id == sp.metadata.id:
                wos_client.subscriptions.delete(sub.metadata.id, background_mode=False)
        wos_client.service_providers.delete(sp.metadata.id, background_mode=False)
        print("Deleted existing provider", sp.metadata.id)

service_provider_id = wos_client.service_providers.add(
    name=SERVICE_PROVIDER_NAME,
    description="SageMaker endpoints for the hiring demo",
    service_type=ServiceTypes.AMAZON_SAGEMAKER,
    operational_space_id="production",
    credentials=SageMakerCredentials(
        access_key_id=OPENSCALE_AWS_ACCESS_KEY_ID,
        secret_access_key=OPENSCALE_AWS_SECRET_ACCESS_KEY,
        region=AWS_REGION),
    background_mode=False).result.metadata.id
print("Service provider:", service_provider_id)




 Waiting for end of deleting subscription 01a106ce-d576-7da1-a405-bceec9f74ab5 




finished

---------------------------------------------
 Successfully finished deleting subscription 
---------------------------------------------





 Waiting for end of deleting service provider 01a106ce-9baf-7aeb-83f3-3f2a6796909d 




finished

-------------------------------------------------
 Successfully finished deleting service provider 
-------------------------------------------------


Deleted existing provider 01a106ce-9baf-7aeb-83f3-3f2a6796909d



 Waiting for end of adding service provider 01a10709-3f2e-75e7-8789-cd34aa48095d 




active

-----------------------------------------------
 Successfully finished adding service provider 
-----------------------------------------------


Service provider: 01a10709-3f2e-75e7-8789-cd34aa48095d


## 7. Find the endpoint and compute training statistics

In [12]:
assets = wos_client.service_providers.list_assets(
    data_mart_id=data_mart_id, service_provider_id=service_provider_id).result["resources"]
print("Endpoints visible to OpenScale:", [a["entity"]["name"] for a in assets])

model_asset = next(a for a in assets if a["entity"]["name"] == ENDPOINT_NAME)
model_asset["entity"]

Endpoints visible to OpenScale: ['hiring-model-endpoint-v3']


{'name': 'hiring-model-endpoint-v3',
 'deployment_rn': 'arn:aws:sagemaker:us-east-1:844183293104:endpoint/hiring-model-endpoint-v3',
 'type': 'online',
 'scoring_endpoint': {'url': 'hiring-model-endpoint-v3'},
 'asset': {'asset_id': '63597c9af6fbbbbd853c14a24df47f4a',
  'asset_rn': 'arn:aws:sagemaker:us-east-1:844183293104:model/hiring-model-endpoint-v3-20261004-210141',
  'url': 's3://sagemaker-us-east-1-844183293104/hiring-demo/model/model.tar.gz',
  'name': 'hiring-model-endpoint-v3-20261004-210141',
  'asset_type': 'model',
  'created_at': '2026-10-04T13:01:43.066Z'},
 'asset_properties': {'asset_revision': '1791119094415'}}

In [13]:
# No IBM COS on AWS, so training statistics are computed locally and passed in.
# They drive the fairness and explainability configuration.
from ibm_watson_openscale.utils.training_stats import TrainingStats

training_data_info = {
    "label_column": LABEL,
    "prediction_column": "predicted_label",   # Linear Learner output fields
    "probability_column": "score",
    "feature_columns": FEATURES,
    "categorical_columns": CATEGORICAL,
    "problem_type": "binary",
    "fairness_inputs": {
        "fairness_attributes": [
            {"feature": PROTECTED_ATTRIBUTE, "majority": [[0, 0]], "minority": [[1, 1]], "threshold": 0.8},
        ],
        "favourable_class": [1.0],
        "unfavourable_class": [0.0],
        "min_records": 100,
    },
}
training_stats = TrainingStats(train_df, training_data_info, explain=True, fairness=True).get_training_statistics()
print(list(training_stats.keys()))

['common_configuration', 'explainability_configuration', 'fairness_configuration']


## 8. Subscribe the SageMaker endpoint

In [14]:
for sub in wos_client.subscriptions.list().result.subscriptions:
    if sub.entity.deployment.name == ENDPOINT_NAME:
        wos_client.subscriptions.delete(sub.metadata.id, background_mode=False)
        print("Deleted existing subscription", sub.metadata.id)

common = training_stats["common_configuration"]
input_fields = [f for f in common["input_data_schema"]["fields"] if f["name"] != LABEL]

subscription_id = wos_client.subscriptions.add(
    data_mart_id=data_mart_id,
    service_provider_id=service_provider_id,
    asset=Asset(
        asset_id=model_asset["entity"]["asset"]["asset_id"],
        name=model_asset["entity"]["asset"]["name"],
        url=model_asset["entity"]["asset"]["url"],
        asset_type=model_asset["entity"]["asset"].get("asset_type", "model"),
        problem_type=ProblemType.BINARY_CLASSIFICATION,
        input_data_type=InputDataType.STRUCTURED),
    deployment=AssetDeploymentRequest(
        deployment_id=model_asset["metadata"]["guid"],
        url=model_asset["metadata"]["url"],
        name=model_asset["entity"]["name"],
        deployment_type=model_asset["entity"]["type"],
        scoring_endpoint=ScoringEndpointRequest(url=model_asset["entity"]["scoring_endpoint"]["url"])),
    asset_properties=AssetPropertiesRequest(
        label_column=LABEL,
        prediction_field="predicted_label",
        probability_fields=["score"],
        feature_fields=FEATURES,
        categorical_fields=CATEGORICAL,
        input_data_schema=SparkStruct.from_dict({"type": "struct", "fields": input_fields}),
        training_data_schema=SparkStruct.from_dict(common["training_data_schema"]),
        output_data_schema=SparkStruct.from_dict(common["output_data_schema"])),
    background_mode=False).result.metadata.id
print("Subscription:", subscription_id)




 Waiting for end of adding subscription 01a10709-6452-7105-a902-42788ed549b3 




active

-------------------------------------------
 Successfully finished adding subscription 
-------------------------------------------


Subscription: 01a10709-6452-7105-a902-42788ed549b3


## 9. Payload logging (batch 1)

In [15]:
import uuid
import time
from ibm_watson_openscale.supporting_classes.payload_record import PayloadRecord

time.sleep(5)
payload_data_set_id = wos_client.data_sets.list(
    type=DataSetTypes.PAYLOAD_LOGGING, target_target_id=subscription_id,
    target_target_type=TargetTypes.SUBSCRIPTION).result.data_sets[0].metadata.id

def log_payload(df, batch_size=50):
    """Score df on SageMaker and log each batch to the OpenScale payload table."""
    for i in range(0, len(df), batch_size):
        chunk = df.iloc[i:i + batch_size]
        preds, ms = score(chunk)
        wos_client.data_sets.store_records(
            data_set_id=payload_data_set_id, background_mode=False,
            request_body=[PayloadRecord(
                scoring_id=str(uuid.uuid4()),
                request={"fields": FEATURES, "values": chunk[FEATURES].values.tolist()},
                response={"fields": ["predicted_label", "score"],
                          "values": [[p["predicted_label"], [1 - p["score"], p["score"]]] for p in preds]},
                response_time=ms)])
    time.sleep(5)
    print("Payload records:", wos_client.data_sets.get_records_count(payload_data_set_id))

# The less biased batch goes first so fairness declines between the two evaluations
log_payload(payload_df_2)




 Waiting for end of storing records with request id: 06fefdaf-8b63-43f1-9835-f15c09803fed 




active

---------------------------------------
 Successfully finished storing records 
---------------------------------------





 Waiting for end of storing records with request id: 68adafe3-c718-497a-84e6-8fed8b22d128 




active

---------------------------------------
 Successfully finished storing records 
---------------------------------------





 Waiting for end of storing records with request id: 0b8b481a-42dd-4a7c-b91c-01c9e42c7ba8 




active

---------------------------------------
 Successfully finished storing records 
---------------------------------------


Payload records: 150


## 10. Quality monitor and feedback

In [16]:
target = Target(target_type=TargetTypes.SUBSCRIPTION, target_id=subscription_id)

quality_monitor_id = wos_client.monitor_instances.create(
    data_mart_id=data_mart_id,
    monitor_definition_id=wos_client.monitor_definitions.MONITORS.QUALITY.ID,
    target=target,
    parameters={"min_feedback_data_size": 50},
    thresholds=[{"metric_id": "area_under_roc", "type": "lower_limit", "value": 0.70}],
    background_mode=False).result.metadata.id

time.sleep(10)
feedback_data_set_id = wos_client.data_sets.list(
    type=DataSetTypes.FEEDBACK, target_target_id=subscription_id,
    target_target_type=TargetTypes.SUBSCRIPTION).result.data_sets[0].metadata.id

feedback_cols = FEATURES + [LABEL]
wos_client.data_sets.store_records(
    feedback_data_set_id,
    request_body=[{"fields": feedback_cols, "values": feedback_df[feedback_cols].values.tolist()}],
    background_mode=False)
print("Feedback records:", wos_client.data_sets.get_records_count(feedback_data_set_id))




 Waiting for end of monitor instance creation 01a1070a-0684-749c-b604-eeff4a03820f 




preparing
active

---------------------------------------
 Monitor instance successfully created 
---------------------------------------





 Waiting for end of storing records with request id: 4672a7c1-e061-4703-a394-09130134735c 




pending
active

---------------------------------------
 Successfully finished storing records 
---------------------------------------


Feedback records: 110


## 11. Fairness and explainability monitors (from training statistics)

In [17]:
ids = wos_client.monitor_definitions.MONITORS

# With training_data_stats the SDK creates both monitors from the stats; the first three
# arguments are still required by the method signature.
monitors = wos_client.monitor_instances.create(
    monitor_definition_id=ids.FAIRNESS.ID,
    target=Target(target_type=TargetTypes.SUBSCRIPTION, target_id=subscription_id),
    parameters={},
    data_mart_id=data_mart_id,
    training_data_stats=training_stats,
    subscription_id=subscription_id,
    background_mode=False)

fairness_monitor_id = monitors[ids.FAIRNESS.ID]["metadata"]["id"]
explainability_monitor_id = monitors[ids.EXPLAINABILITY.ID]["metadata"]["id"]
print("Fairness:", fairness_monitor_id, "| Explainability:", explainability_monitor_id)

Creating fairness monitor



 Waiting for end of monitor instance creation 01a1070a-8f37-748d-999c-2979e9a157a9 




active

---------------------------------------
 Monitor instance successfully created 
---------------------------------------


Creating Explain monitor



 Waiting for end of monitor instance creation 01a1070a-a92d-7f1a-a51e-8fde840b3b51 




preparing
active

---------------------------------------
 Monitor instance successfully created 
---------------------------------------


Fairness: 01a1070a-8f37-748d-999c-2979e9a157a9 | Explainability: 01a1070a-a92d-7f1a-a51e-8fde840b3b51


## 12. Run evaluations

In [18]:
refresh_token()
wos_client.monitor_instances.run(monitor_instance_id=quality_monitor_id, background_mode=False)
wos_client.monitor_instances.show_metrics(monitor_instance_id=quality_monitor_id)




 Waiting for end of monitoring run 8f6af96e-7c9d-4d0f-946d-2c666a35d197 




finished

---------------------------
 Successfully finished run 
---------------------------




2026-10-04 13:11:55.206000+00:00,area_under_roc,01a1070a-de46-7293-aaac-92adf7f31f6b,0.7304761904761904,0.7,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:55.206000+00:00,area_under_pr,01a1070a-de46-7293-aaac-92adf7f31f6b,0.6967532467532468,None,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:55.206000+00:00,accuracy,01a1070a-de46-7293-aaac-92adf7f31f6b,0.8090909090909091,None,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:55.206000+00:00,true_positive_rate,01a1070a-de46-7293-aaac-92adf7f31f6b,0.5142857142857142,None,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:55.206000+00:00,false_positive_rate,01a1070a-de46-7293-aaac-92adf7f31f6b,0.05333333333333334,None,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:55.206000+00:00,recall,01a1070a-de46-7293-aaac-92adf7f31f6b,0.5142857142857142,None,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:55.206000+00:00,precision,01a1070a-de46-7293-aaac-92adf7f31f6b,0.8181818181818182,None,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:55.206000+00:00,f1_measure,01a1070a-de46-7293-aaac-92adf7f31f6b,0.6315789473684209,None,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:55.206000+00:00,log_loss,01a1070a-de46-7293-aaac-92adf7f31f6b,0.36481678578750626,None,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:55.206000+00:00,brier_score,01a1070a-de46-7293-aaac-92adf7f31f6b,0.6300101615317181,None,None,['model_type:original'],quality,01a1070a-0684-749c-b604-eeff4a03820f,8f6af96e-7c9d-4d0f-946d-2c666a35d197,subscription,01a10709-6452-7105-a902-42788ed549b3


Note: First 10 records were displayed.


In [19]:
# Creating the fairness monitor already evaluated the first payload batch, and a new run needs payload
# records it has not seen yet (otherwise it fails with AIQFM6009), so only show the metrics
# here. Fairness is evaluated again after the second payload batch in section 14.
wos_client.monitor_instances.show_metrics(monitor_instance_id=fairness_monitor_id)

2026-10-04 13:11:52.126107+00:00,fairness_value,01a1070a-d23e-77de-ac51-3136f03531a4,58.667,80.0,None,"['feature:IsFemale', 'feature_value:1-1', 'fairness_metric_type:fairness']",fairness,01a1070a-8f37-748d-999c-2979e9a157a9,eec8a7c8-d534-4257-9701-45dc256d4eca,subscription,01a10709-6452-7105-a902-42788ed549b3


In [20]:
# Model health (latency, throughput, record counts) is only evaluated hourly, and its first
# scheduled run happens before any payload is logged. Run it now so the metrics show in the UI.
model_health_monitor_id = wos_client.monitor_instances.list(
    target_target_id=subscription_id,
    monitor_definition_id="model_health").result.monitor_instances[0].metadata.id
wos_client.monitor_instances.run(monitor_instance_id=model_health_monitor_id, background_mode=False)
wos_client.monitor_instances.show_metrics(monitor_instance_id=model_health_monitor_id)




 Waiting for end of monitoring run 85a3e9cc-3f58-49e0-85e3-62d6afd6a187 




finished

---------------------------
 Successfully finished run 
---------------------------




2026-10-04 13:12:04.779441+00:00,minimum_api_latency,01a1070b-03ab-7519-ae6a-fa82d1f1c738,270.0,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:12:04.779441+00:00,average_api_latency,01a1070b-03ab-7519-ae6a-fa82d1f1c738,698.0,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:12:04.779441+00:00,median_api_latency,01a1070b-03ab-7519-ae6a-fa82d1f1c738,272.0,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:12:04.779441+00:00,maximum_api_latency,01a1070b-03ab-7519-ae6a-fa82d1f1c738,1552.0,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:12:04.779441+00:00,minimum_api_throughput,01a1070b-03ab-7519-ae6a-fa82d1f1c738,0.644,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:12:04.779441+00:00,average_api_throughput,01a1070b-03ab-7519-ae6a-fa82d1f1c738,1.433,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:12:04.779441+00:00,median_api_throughput,01a1070b-03ab-7519-ae6a-fa82d1f1c738,3.676,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:12:04.779441+00:00,maximum_api_throughput,01a1070b-03ab-7519-ae6a-fa82d1f1c738,3.704,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:12:04.779441+00:00,minimum_record_latency,01a1070b-03ab-7519-ae6a-fa82d1f1c738,5.4,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:12:04.779441+00:00,average_record_latency,01a1070b-03ab-7519-ae6a-fa82d1f1c738,13.96,None,None,[],model_health,01a10709-6737-7189-8ccb-553ed0a68f3b,85a3e9cc-3f58-49e0-85e3-62d6afd6a187,subscription,01a10709-6452-7105-a902-42788ed549b3


Note: First 10 records were displayed.


## 13. Explain individual hiring decisions

OpenScale calls the SageMaker endpoint with perturbed inputs (LIME) using the provider's access key.

In [21]:
records = wos_client.data_sets.get_list_of_records(
    data_set_id=payload_data_set_id, limit=3, output_type="pandas").result
scoring_ids = records["scoring_id"].tolist()

task = wos_client.monitor_instances.explanation_tasks(
    scoring_ids=scoring_ids, explanation_types=["lime", "contrastive"],
    subscription_id=subscription_id).result
explanation_task_id = task.to_dict()["metadata"]["explanation_task_ids"][0]

time.sleep(30)
wos_client.monitor_instances.get_explanation_tasks(
    explanation_task_id=explanation_task_id, subscription_id=subscription_id).result.to_dict()

{'metadata': {'explanation_task_id': 'ccab500f-f9c4-43f9-a694-a15c36ded9de',
  'created_by': 'afd6f60f-f4cd-425a-a40f-98b6d9170c39',
  'created_at': '2026-10-04T13:12:11.658080Z',
  'updated_at': '2026-10-04T13:12:13.939082Z'},
 'entity': {'status': {'state': 'in_progress'},
  'asset': {'id': '63597c9af6fbbbbd853c14a24df47f4a',
   'name': 'hiring-model-endpoint-v3-20261004-210141',
   'input_data_type': 'structured',
   'problem_type': 'binary',
   'deployment': {'id': '5a74e91c248cc6df305419a56c627c1a',
    'name': 'hiring-model-endpoint-v3'}},
  'input_features': [{'name': 'Age',
    'value': 51.0,
    'feature_type': 'numerical'},
   {'name': 'BusinessTravel', 'value': 1.0, 'feature_type': 'numerical'},
   {'name': 'Education', 'value': 2.0, 'feature_type': 'numerical'},
   {'name': 'RelevantEducationLevel',
    'value': 3.0,
    'feature_type': 'numerical'},
   {'name': 'JobLevel', 'value': 2.0, 'feature_type': 'numerical'},
   {'name': 'MaritalStatus', 'value': 1.0, 'feature_type'

## 14. Drift v2 and the second payload batch

Drift v2 compares production payload with a baseline built from the training data. There is no IBM COS on
AWS for OpenScale to read the training data from, so the baseline archive is built here (scored with the
local copy of the model), uploaded, and the monitor is evaluated on the first payload batch. The second
batch is then logged and fairness, drift and model health are evaluated again.

In [22]:
from ibm_watson_openscale.utils.configuration_utility import ConfigurationUtility

def run_when_idle(monitor_instance_id, wait_seconds=900):
    """Run a monitor once no other run is in progress on it (a second concurrent run fails with HTTP 429)."""
    deadline = time.time() + wait_seconds
    while time.time() < deadline:
        runs = wos_client.monitor_instances.list_runs(monitor_instance_id=monitor_instance_id).result.runs
        if all(r.entity.status.state in ("finished", "error", "failed", "canceled") for r in runs[:5]):
            break
        print("Waiting for the run already in progress...")
        time.sleep(20)
    refresh_token()
    return wos_client.monitor_instances.run(monitor_instance_id=monitor_instance_id, background_mode=False)

def score_locally(df, schema=None):
    """Same model as the endpoint, scored in the notebook: returns (probabilities, predictions)."""
    proba = clf.predict_proba(scaler.transform(df[FEATURES]))
    return proba, (proba[:, 1] >= 0.5).astype(float)

drift_config = ConfigurationUtility(
    training_data=train_df.copy(),
    common_parameters={
        "asset_type": "model", "problem_type": "binary", "input_data_type": "structured",
        "label_column": LABEL, "prediction_column": "predicted_label", "probability_column": "score",
        "feature_columns": FEATURES, "categorical_columns": CATEGORICAL, "class_labels": [0.0, 1.0],
        "enable_explainability": False, "enable_fairness": False, "enable_drift_v2": True,
    },
    scoring_fn=score_locally)

# Feature importance comes straight from the model's coefficients (takes ~30 seconds overall)
feature_importance = {f: float(abs(c)) for f, c in zip(FEATURES, clf.coef_[0])}
with open("sagemaker_model/drift_v2_baseline.tar.gz", "wb") as f:
    f.write(drift_config.get_drift_v2_archive({"min_samples": 100}, feature_importance=feature_importance))

refresh_token()
wos_client.monitor_instances.upload_drift_v2_archive(
    subscription_id=subscription_id,
    archive_path="sagemaker_model/drift_v2_baseline.tar.gz",
    archive_name="drift_v2_baseline.tar.gz")

drift_monitor_id = wos_client.monitor_instances.create(
    data_mart_id=data_mart_id,
    monitor_definition_id="drift_v2",
    target=Target(target_type=TargetTypes.SUBSCRIPTION, target_id=subscription_id),
    parameters={"min_samples": 100, "train_archive": False},
    background_mode=False).result.metadata.id

# First drift evaluation, on the payload batch logged in section 9. Creating the monitor can
# start a run by itself, and a drift run takes several minutes.
run_when_idle(drift_monitor_id)
wos_client.monitor_instances.show_metrics(monitor_instance_id=drift_monitor_id)

2026-10-04 13:15:29.331001+00:00,confidence_drift_score,01a1070e-22b3-79e6-bda7-9823d8b172d3,0.4472,None,0.05,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 0.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:15:29.331001+00:00,records_processed,01a1070e-22b3-79e6-bda7-9823d8b172d3,150.0,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 0.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:15:29.331001+00:00,confidence_drift_score,01a1070e-22b3-79e6-bda7-9823d8b172d3,0.4485,None,0.05,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 0.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:15:29.331001+00:00,records_processed,01a1070e-22b3-79e6-bda7-9823d8b172d3,150.0,None,None,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 0.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:15:34.895274+00:00,confidence_drift_score,01a1070e-386f-79ff-8ca5-d6c77e0a0dc2,0.4469,None,0.05,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 1.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:15:34.895274+00:00,records_processed,01a1070e-386f-79ff-8ca5-d6c77e0a0dc2,150.0,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 1.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:15:34.895274+00:00,confidence_drift_score,01a1070e-386f-79ff-8ca5-d6c77e0a0dc2,0.4477,None,0.05,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 1.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:15:34.895274+00:00,records_processed,01a1070e-386f-79ff-8ca5-d6c77e0a0dc2,150.0,None,None,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 1.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:15:38.196249+00:00,prediction_drift_score,01a1070e-4554-7089-8e78-e70ccd76fe1a,0.2931,None,0.05,"['algorithm_used:jensen_shannon', 'computed_on:payload', 'field_type:class', 'field_name:predicted_label']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:15:38.196249+00:00,records_processed,01a1070e-4554-7089-8e78-e70ccd76fe1a,150.0,None,None,"['algorithm_used:jensen_shannon', 'computed_on:payload', 'field_type:class', 'field_name:predicted_label']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,3953de3e-223e-4a3f-a4d4-698e9bc157ff,subscription,01a10709-6452-7105-a902-42788ed549b3


In [23]:
refresh_token()
log_payload(payload_df_1)

# New payload records are available now, so fairness can be evaluated again
wos_client.monitor_instances.run(monitor_instance_id=fairness_monitor_id, background_mode=False)
wos_client.monitor_instances.show_metrics(monitor_instance_id=fairness_monitor_id)

# Refresh model health with the new scoring requests
wos_client.monitor_instances.run(monitor_instance_id=model_health_monitor_id, background_mode=False)

# Drift on the second batch
run_when_idle(drift_monitor_id)
wos_client.monitor_instances.show_metrics(monitor_instance_id=drift_monitor_id)

2026-10-04 13:20:08.715290+00:00,fairness_value,01a10712-660b-7ee6-97d9-58b1f676eff8,43.438,80.0,None,"['feature:IsFemale', 'feature_value:1-1', 'fairness_metric_type:fairness']",fairness,01a1070a-8f37-748d-999c-2979e9a157a9,f2b7f27a-d24e-4c11-8d4c-037a9d1d77b7,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:11:52.126107+00:00,fairness_value,01a1070a-d23e-77de-ac51-3136f03531a4,58.667,80.0,None,"['feature:IsFemale', 'feature_value:1-1', 'fairness_metric_type:fairness']",fairness,01a1070a-8f37-748d-999c-2979e9a157a9,eec8a7c8-d534-4257-9701-45dc256d4eca,subscription,01a10709-6452-7105-a902-42788ed549b3


2026-10-04 13:23:45.318109+00:00,confidence_drift_score,01a10715-b426-7e20-ade9-d794dec21f9f,0.4318,None,0.05,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 0.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:23:45.318109+00:00,records_processed,01a10715-b426-7e20-ade9-d794dec21f9f,150.0,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 0.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:23:45.318109+00:00,confidence_drift_score,01a10715-b426-7e20-ade9-d794dec21f9f,0.4331,None,0.05,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 0.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:23:45.318109+00:00,records_processed,01a10715-b426-7e20-ade9-d794dec21f9f,150.0,None,None,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 0.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:23:54.090278+00:00,confidence_drift_score,01a10715-d66a-75da-87a1-c10237ae2a1e,0.4302,None,0.05,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 1.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:23:54.090278+00:00,records_processed,01a10715-d66a-75da-87a1-c10237ae2a1e,150.0,None,None,"['algorithm_used:total_variation', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 1.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:23:54.090278+00:00,confidence_drift_score,01a10715-d66a-75da-87a1-c10237ae2a1e,0.4309,None,0.05,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 1.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:23:54.090278+00:00,records_processed,01a10715-d66a-75da-87a1-c10237ae2a1e,150.0,None,None,"['algorithm_used:overlap_coefficient', 'computed_on:payload', 'field_type:class', 'field_name:Class Probability for 1.0']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:24:00.283705+00:00,prediction_drift_score,01a10715-ee9b-7eaa-98e3-c965ca72e2b4,0.2875,None,0.05,"['algorithm_used:jensen_shannon', 'computed_on:payload', 'field_type:class', 'field_name:predicted_label']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3
2026-10-04 13:24:00.283705+00:00,records_processed,01a10715-ee9b-7eaa-98e3-c965ca72e2b4,150.0,None,None,"['algorithm_used:jensen_shannon', 'computed_on:payload', 'field_type:class', 'field_name:predicted_label']",drift_v2,01a1070c-0d78-7b12-ada1-d2c642a840a3,919026ee-0286-43d0-97d4-b3b173d9e89c,subscription,01a10709-6452-7105-a902-42788ed549b3


## 15. Pre-production validation (same endpoint)

Registers the same SageMaker endpoint a second time under a **pre-production** provider, so the deployment
also appears on the pre-production side of the dashboard (and in the *Validate* stage of an AI use case).
Pre-production does not use payload logging: the model is evaluated against the labelled test file
`hiring_eval_data_1.csv`, which OpenScale scores on the endpoint itself.

In [28]:
PREPROD_PROVIDER_NAME = SERVICE_PROVIDER_NAME + " (pre-production)"

# Remove a previous pre-production provider and its subscriptions; production is left untouched
for sp in wos_client.service_providers.list().result.service_providers:
    if sp.entity.name == PREPROD_PROVIDER_NAME:
        for sub in wos_client.subscriptions.list().result.subscriptions:
            if sub.entity.service_provider_id == sp.metadata.id:
                wos_client.subscriptions.delete(sub.metadata.id, background_mode=False)
        wos_client.service_providers.delete(sp.metadata.id, background_mode=False)
        print("Deleted existing provider", sp.metadata.id)

preprod_provider_id = wos_client.service_providers.add(
    name=PREPROD_PROVIDER_NAME,
    description="Pre-production validation of the hiring endpoint",
    service_type=ServiceTypes.AMAZON_SAGEMAKER,
    operational_space_id="pre_production",
    credentials=SageMakerCredentials(
        access_key_id=OPENSCALE_AWS_ACCESS_KEY_ID,
        secret_access_key=OPENSCALE_AWS_SECRET_ACCESS_KEY,
        region=AWS_REGION),
    background_mode=False).result.metadata.id
print("Pre-production provider:", preprod_provider_id)

In [29]:
preprod_asset = next(
    a for a in wos_client.service_providers.list_assets(
        data_mart_id=data_mart_id, service_provider_id=preprod_provider_id).result["resources"]
    if a["entity"]["name"] == ENDPOINT_NAME)

common = training_stats["common_configuration"]
input_fields = [f for f in common["input_data_schema"]["fields"] if f["name"] != LABEL]

preprod_subscription_id = wos_client.subscriptions.add(
    data_mart_id=data_mart_id,
    service_provider_id=preprod_provider_id,
    asset=Asset(
        asset_id=preprod_asset["entity"]["asset"]["asset_id"],
        name=preprod_asset["entity"]["asset"]["name"],
        url=preprod_asset["entity"]["asset"]["url"],
        asset_type=preprod_asset["entity"]["asset"].get("asset_type", "model"),
        problem_type=ProblemType.BINARY_CLASSIFICATION,
        input_data_type=InputDataType.STRUCTURED),
    deployment=AssetDeploymentRequest(
        deployment_id=preprod_asset["metadata"]["guid"],
        url=preprod_asset["metadata"]["url"],
        name=preprod_asset["entity"]["name"],
        deployment_type=preprod_asset["entity"]["type"],
        scoring_endpoint=ScoringEndpointRequest(url=preprod_asset["entity"]["scoring_endpoint"]["url"])),
    asset_properties=AssetPropertiesRequest(
        label_column=LABEL,
        prediction_field="predicted_label",
        probability_fields=["score"],
        feature_fields=FEATURES,
        categorical_fields=CATEGORICAL,
        input_data_schema=SparkStruct.from_dict({"type": "struct", "fields": input_fields}),
        training_data_schema=SparkStruct.from_dict(common["training_data_schema"]),
        output_data_schema=SparkStruct.from_dict(common["output_data_schema"])),
    background_mode=False).result.metadata.id
print("Pre-production subscription:", preprod_subscription_id)

In [30]:
# Same monitors and thresholds as production, so the two stages are comparable
preprod_target = Target(target_type=TargetTypes.SUBSCRIPTION, target_id=preprod_subscription_id)
ids = wos_client.monitor_definitions.MONITORS

preprod_quality_monitor_id = wos_client.monitor_instances.create(
    data_mart_id=data_mart_id,
    monitor_definition_id=ids.QUALITY.ID,
    target=preprod_target,
    parameters={"min_feedback_data_size": 50},
    thresholds=[{"metric_id": "area_under_roc", "type": "lower_limit", "value": 0.70}],
    background_mode=False).result.metadata.id

preprod_monitors = wos_client.monitor_instances.create(
    monitor_definition_id=ids.FAIRNESS.ID,
    target=preprod_target,
    parameters={},
    data_mart_id=data_mart_id,
    training_data_stats=training_stats,
    subscription_id=preprod_subscription_id,
    background_mode=False)
preprod_fairness_monitor_id = preprod_monitors[ids.FAIRNESS.ID]["metadata"]["id"]

# The risk-evaluation ("mrm") monitor runs the pre-production test; reuse it if the
# subscription already has one.
mrm_instances = wos_client.monitor_instances.list(
    data_mart_id=data_mart_id, monitor_definition_id="mrm",
    target_target_id=preprod_subscription_id).result.monitor_instances
if mrm_instances:
    mrm_monitor_id = mrm_instances[0].metadata.id
else:
    mrm_monitor_id = wos_client.monitor_instances.create(
        data_mart_id=data_mart_id,
        monitor_definition_id="mrm",
        target=preprod_target,
        parameters={},
        background_mode=False).result.metadata.id
print("Quality:", preprod_quality_monitor_id, "| Fairness:", preprod_fairness_monitor_id, "| MRM:", mrm_monitor_id)

In [31]:
# Upload the labelled test data; OpenScale scores it on the endpoint and evaluates all monitors
refresh_token()
wos_client.monitor_instances.mrm.evaluate_risk(
    monitor_instance_id=mrm_monitor_id,
    test_data_set_name="hiring_eval_data_1.csv",
    test_data_path=f"{DATA_DIR}/hiring_eval_data_1.csv",
    content_type="text/csv",
    includes_model_output="false",
    publish_fact="true",   # send the results to the model's factsheet
    background_mode=False)
# The call above does not report a failed evaluation, so show its status (and the reason if it failed)
print("Risk evaluation:", wos_client.monitor_instances.mrm.get_risk_evaluation(
    monitor_instance_id=mrm_monitor_id).result.entity.status.to_dict())

wos_client.monitor_instances.show_metrics(monitor_instance_id=preprod_quality_monitor_id)
wos_client.monitor_instances.show_metrics(monitor_instance_id=preprod_fairness_monitor_id)

2026-10-04 13:32:04.698000+00:00,area_under_roc,01a1071d-52da-793f-8515-d87f6779cc6a,0.7304761904761904,0.7,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d
2026-10-04 13:32:04.698000+00:00,area_under_pr,01a1071d-52da-793f-8515-d87f6779cc6a,0.6967532467532468,None,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d
2026-10-04 13:32:04.698000+00:00,accuracy,01a1071d-52da-793f-8515-d87f6779cc6a,0.8090909090909091,None,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d
2026-10-04 13:32:04.698000+00:00,true_positive_rate,01a1071d-52da-793f-8515-d87f6779cc6a,0.5142857142857142,None,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d
2026-10-04 13:32:04.698000+00:00,false_positive_rate,01a1071d-52da-793f-8515-d87f6779cc6a,0.05333333333333334,None,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d
2026-10-04 13:32:04.698000+00:00,recall,01a1071d-52da-793f-8515-d87f6779cc6a,0.5142857142857142,None,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d
2026-10-04 13:32:04.698000+00:00,precision,01a1071d-52da-793f-8515-d87f6779cc6a,0.8181818181818182,None,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d
2026-10-04 13:32:04.698000+00:00,f1_measure,01a1071d-52da-793f-8515-d87f6779cc6a,0.6315789473684209,None,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d
2026-10-04 13:32:04.698000+00:00,log_loss,01a1071d-52da-793f-8515-d87f6779cc6a,0.36481678578750615,None,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d
2026-10-04 13:32:04.698000+00:00,brier_score,01a1071d-52da-793f-8515-d87f6779cc6a,0.6300101615317183,None,None,['model_type:original'],quality,01a1071c-753c-75d3-ae4f-ba16793bb947,98b4a07e-2d3f-4980-b733-aa9296a92459,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d


2026-10-04 13:32:12.599816+00:00,fairness_value,01a1071d-71b7-7950-8aba-4908c4b3ab71,12.986,80.0,None,"['feature:IsFemale', 'feature_value:1-1', 'fairness_metric_type:fairness']",fairness,01a1071c-a2bc-73c2-80b0-5008c482dfba,0c529e20-833d-4ec0-af46-0230df087f53,subscription,01a1071c-5a23-7817-adc9-114e8bd8f90d


## 16. Place the deployment in the AI use case lifecycle (only if needed)

Once the model is tracked and evaluated, it should move to **Validation** or **Operation** by itself. If it is
still under **Development** with *Pending evaluation*, this cell moves its deployment to the right stage and
re-runs the matching OpenScale evaluation. It needs sections 3b and 4.

Set `LIFECYCLE_STAGE` to `validate` (the pre-production subscription) or `operate` (the production one).

In [32]:
LIFECYCLE_STAGE = "validate"   # "validate" (pre-production) or "operate" (production)

deployment = next(d for d in external_model.get_deployments() if d.get_name() == ENDPOINT_NAME)
# Moving a deployment to the stage it is already in is rejected, so only move it when needed
current_stage = (external_model.get_environment_type().get("classification") or "").lower()
if current_stage != LIFECYCLE_STAGE:
    deployment.set_environment_type(to_container=LIFECYCLE_STAGE)

# The endpoint is subscribed once per stage; pick the subscription that belongs to this one
refresh_token()
space = {"validate": "pre_production", "operate": "production"}[LIFECYCLE_STAGE]
stage_provider_ids = {sp.metadata.id for sp in wos_client.service_providers.list().result.service_providers
                      if sp.entity.operational_space_id == space}
stage_subscription_id = next(
    s.metadata.id for s in wos_client.subscriptions.list().result.subscriptions
    if s.entity.deployment.name == ENDPOINT_NAME and s.entity.service_provider_id in stage_provider_ids)

def stage_monitor_id(definition_id):
    return wos_client.monitor_instances.list(
        target_target_id=stage_subscription_id,
        monitor_definition_id=definition_id).result.monitor_instances[0].metadata.id

# Evaluate again now that the external model exists, so OpenScale publishes the results to it
if LIFECYCLE_STAGE == "validate":
    wos_client.monitor_instances.mrm.evaluate_risk(
        monitor_instance_id=stage_monitor_id("mrm"),
        test_data_set_name="hiring_eval_data_1.csv",
        test_data_path=f"{DATA_DIR}/hiring_eval_data_1.csv",
        content_type="text/csv",
        includes_model_output="false",
        publish_fact="true",   # send the results to the model's factsheet
        background_mode=False)
    # The call above does not report a failed evaluation, so show its status (and the reason if it failed)
    print("Risk evaluation:", wos_client.monitor_instances.mrm.get_risk_evaluation(
        monitor_instance_id=stage_monitor_id("mrm")).result.entity.status.to_dict())
else:
    wos_client.monitor_instances.run(monitor_instance_id=stage_monitor_id("quality"), background_mode=False)

external_model.get_environment_type()

{'classification': 'VALIDATE',
 'reason': 'The space is marked as pre-production by Watson Open Scale which is considered as PRE-PRODUCTION environment and asset shows under VALIDATE stage'}

In [33]:
status = wos_client.monitor_instances.mrm.get_risk_evaluation(
    monitor_instance_id=stage_monitor_id("mrm")).result.entity.status.to_dict()
print("Risk evaluation:", status, flush=True)
status


{'state': 'finished',
 'queued_at': '2026-10-04T13:34:02.789000Z',
 'started_at': '2026-10-04T13:34:03.125000Z',
 'updated_at': '2026-10-04T13:34:17.907000Z',
 'completed_at': '2026-10-04T13:34:16.508000Z',
 'message': 'Mrm evaluation complete.',
 'operators': []}

## 17. Show it in the UI

Open `https://<region>.aws.data.ibm.com/aiopenscale/insights`, select **hiring-model-endpoint** (listed under both production and pre-production), and walk through
Quality, Fairness (`IsFemale`), Drift v2 and the *Explain* view on individual transactions.

## 18. Clean up (optional)

In [ ]:
# wos_client.subscriptions.delete(subscription_id, background_mode=False)
# wos_client.service_providers.delete(service_provider_id, background_mode=False)
# wos_client.subscriptions.delete(preprod_subscription_id, background_mode=False)
# wos_client.service_providers.delete(preprod_provider_id, background_mode=False)
# sm.delete_endpoint(EndpointName=ENDPOINT_NAME)   # stops SageMaker endpoint charges